# 15 — Executar e verificar a avaliação da referência

Autor: Wanderson Ferreira. Preparação: 2026-10-09.

**HGB = Histogram-based Gradient Boosting**, modelo de abril já congelado.
Implementação incremental sobre a etapa 14; nenhuma nova seleção de modelo.
Esta preparação usa somente dados sintéticos. A execução nativa de setembro
continua pendente e seus resultados não são antecipados neste notebook.


## Problema, alternativas e decisão

A janela foi fixada antes dos resultados. Precisamos verificar o modelo correto,
respeitar o histórico de rótulos, conservar todos os dias e registrar a abertura
da reserva. Reexecutar o pipeline original abriria períodos fora dessa autorização;
transformar a CLI de plano em avaliação também mudaria seu compromisso anterior.

Mantemos o plano de metadados e adicionamos um executor explícito com outro
snapshot de autorização. Reutilizamos o carregador de MLflow/skops, o SQL causal
e a interface de scores. O builder de dados fica em `data/datasets/`, as métricas
em `evaluation/` e a composição/auditoria em `modeling/experiments/`.

A assinatura opcional de raiz no carregador congelado permite apontar para o store
correto sem alterar o comportamento dos executores existentes. Não duplicamos
um trainer ou criamos outra implementação de features.


## Acesso e recuperação

| Fase | Compromisso |
| --- | --- |
| Conferir metadados | Inputs/código commitados, ambiente e fonte fixados |
| Carregar referência | Conferir bytes/signatura/parâmetros antes das partições |
| Construir features | Registrar acesso; ler somente 19/08–15/09 |
| Pontuar | Usar as 19 features dos 14 dias, sem fit ou novas runs |
| Publicar | Bundle atômico, relatório, predições, features e hashes |
| Verificar/retomar | Recalcular resultados salvos; não reabrir fonte/modelo quando completo |

`run` consome a reserva de qualidade de 02–15/09. Os dados não serão usados como
confirmação independente de hipóteses orientadas pelos novos resultados. O replay
16–30/09 continua fechado, e a fila permanece retrospectiva por dia completo.


In [ ]:
# Só metadados versionados; não executa a avaliação.
import json

from fraud_detection_mlops.config import PROJECT_ROOT

execution = json.loads(
    (PROJECT_ROOT / "references/reference_assessment_execution_v1.json").read_text()
)
execution["version"], execution["access_policy"]

## Resultados e evidência dos checks

Os cenários sintéticos percorrem 28 partições de contexto/avaliação, preservando
os 14 dias e verificando o histórico conhecido por máscara temporal independente.
Partições inválidas em maio e no replay não são abertas. Dias de uma classe
conservam contagens e desfechos indefinidos explicitamente nulos.

Conferimos bloqueio por código não commitado, dados inválidos, symlinks,
corrupção rehasheada, concorrência e retomada após interrupção da publicação.
A carga de modelo usa também MLflow/skops reais em um store temporário; não cria
outra run nem ajusta durante a carga. Isso é evidência da implementação, não da
qualidade do modelo nos dados do autor. A contagem e o ambiente da suíte completa
ficam no recibo, evitando replicar dumps de terminal.


In [ ]:
# Consulta apenas um bundle já publicado pelo executor; sem Silver ou MLflow.
from fraud_detection_mlops.modeling.experiments.reference_assessment_execution import (
    verify_assessment,
)

assessment_path = (
    PROJECT_ROOT
    / "data/processed/handbook"
    / "6e67dbd0a3bfe0d7ec33abc4bce5f37cd4ff0d6a"
    / execution["version"]
)
if (assessment_path / "results/manifest.json").is_file():
    saved = verify_assessment(assessment_path)
    print(
        {
            key: saved["metrics"][key]
            for key in ("rows", "average_precision", "daily_customer_precision_at_100")
        }
    )
else:
    print("Avaliação nativa ainda não publicada neste checkout.")

## Limites e próximo passo

A referência de fila aleatória condiciona nos rótulos/populações observados.
Seu intervalo não é incerteza de desempenho futuro, e sua cauda não autoriza
superioridade ou promoção. O executor não adiciona gate confirmatório. A maturação
dos rótulos segue o relógio histórico: todos os rótulos de 02–15/09 são avaliáveis
antes de 23/09, mantendo disponibilidade causal das features em cada transação.

Verificação offline confirma integridade e coerência da evidência salva; não
recalcula scores com o modelo nem autentica externamente toda a execução. Restore
preserva o bundle e a auditoria juntos; mudar código/inputs bloqueia a retomada.

Próximo passo: execução no Alienware, revisão dos resultados e registro do relato;
depois exportação real, Docker e um fluxo pequeno de Prefect. O streaming terá
contrato próprio para eventos, estado, feedback atrasado e política de alertas.

[Comandos e recuperação](../../docs/modeling/EVALUATION_PROTOCOL.md#executor-auditável-da-janela-fixada),
[recibo](../../references/evidence/reference_assessment_execution_preparation_2026-10-09.json),
[mural](../../docs/project/ROADMAP.md#próxima-entrega-concreta).
